In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.append(str(Path('../src').resolve()))
from transformer import AESTransformer

RAW_DIR = Path('../Data/raw/aes_api')
transformer = AESTransformer()

print("--- MISSION START: BATCH TRANSFORMATION ---")

# Find all team directories
team_folders = [d for d in RAW_DIR.iterdir() if d.is_dir() and d.name.startswith("team_")]
total_teams = len(team_folders)

print(f"Identified {total_teams} team folders for processing.\n")

for i, folder in enumerate(team_folders, 1):
    try:
        # Extract the integer ID from the folder name
        team_id = int(folder.name.split('_')[1])
        print(f"[{i}/{total_teams}] Processing Team ID: {team_id}")
        
        # Execute the transformation (saves to ../Data/processed/teams/)
        transformer.process_all(team_id)
        
    except Exception as e:
        print(f"[!] Failed to process folder {folder.name}: {e}")

print("\n--- BATCH TRANSFORMATION COMPLETE ---")


In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_TEAMS_DIR = Path('../Data/processed/teams')
OUTPUT_DIR = Path('../Data/processed')

print("--- MISSION START: DATA AGGREGATION ---")

def aggregate_files(file_suffix, output_filename):
    """Finds all files with a specific suffix, merges them, and prepends the TeamId."""
    files = list(PROCESSED_TEAMS_DIR.glob(f"*{file_suffix}"))
    
    if not files:
        print(f"[-] No files found for suffix: {file_suffix}")
        return
        
    print(f"Aggregating {len(files)} files for {output_filename}...")
    
    dataframes = []
    for f in files:
        try:
            df = pd.read_csv(f)
            if not df.empty:
                # Extract TeamId from filename (e.g., team_185651_matches.csv)
                team_id = f.name.split('_')[1]
                df.insert(0, 'TeamId', team_id)
                dataframes.append(df)
        except Exception as e:
            print(f"Error reading {f.name}: {e}")
            
    if dataframes:
        master_df = pd.concat(dataframes, ignore_index=True)
        out_path = OUTPUT_DIR / output_filename
        master_df.to_csv(out_path, index=False)
        print(f"[SUCCESS] Saved {len(master_df)} rows to {out_path.name}")
    else:
        print(f"[-] All files for {file_suffix} were empty.")

# Execute the Roll-up
aggregate_files('_matches.csv', 'aes_master_matches.csv')
aggregate_files('_finishes.csv', 'aes_master_finishes.csv')
aggregate_files('_roster.csv', 'aes_master_rosters.csv')

print("\n--- AGGREGATION COMPLETE ---")


In [ ]:
import sys
import pandas as pd
from pathlib import Path

sys.path.append(str(Path('../src').resolve()))
from transformer import AESTransformer

# Force reload the module to get the updated code
if 'transformer' in sys.modules:
    del sys.modules['transformer']
from transformer import AESTransformer

RAW_DIR = Path('../Data/raw/aes_api')
PROCESSED_TEAMS_DIR = Path('../Data/processed/teams')
OUTPUT_DIR = Path('../Data/processed')

transformer = AESTransformer()
team_folders = [d for d in RAW_DIR.iterdir() if d.is_dir() and d.name.startswith("team_")]

print("--- MISSION START: FINISHES CATCH-UP ---")
print(f"Processing finishes for {len(team_folders)} teams...")

# 1. Process only the finishes
for folder in team_folders:
    try:
        team_id = int(folder.name.split('_')[1])
        transformer.transform_finishes(team_id)
    except Exception as e:
        pass # Silently skip errors for this quick catch-up

# 2. Aggregate just the finishes
files = list(PROCESSED_TEAMS_DIR.glob("*_finishes.csv"))
print(f"\nAggregating {len(files)} finishes files...")

dataframes = []
for f in files:
    try:
        df = pd.read_csv(f)
        if not df.empty:
            team_id = f.name.split('_')[1]
            df.insert(0, 'TeamId', team_id)
            dataframes.append(df)
    except:
        pass
        
if dataframes:
    master_df = pd.concat(dataframes, ignore_index=True)
    out_path = OUTPUT_DIR / 'aes_master_finishes.csv'
    master_df.to_csv(out_path, index=False)
    print(f"[SUCCESS] Saved {len(master_df)} rows to {out_path.name}")
else:
    print("[-] No data found.")


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TARGET_TEAM_ID = 185651
AES_MATCHES_FILE = Path('../Data/processed/aes_master_matches.csv')
LOCAL_MATCHES_FILE = Path('../Data/processed/events/master_match_results.csv')

print(f"--- MISSION START: COMMON OPPONENT MATRIX (TARGET: {TARGET_TEAM_ID}) ---")

# 1. Build an ID-to-Name Dictionary
# We use your local offline database to map the integer IDs back to human-readable names
df_local = pd.read_csv(LOCAL_MATCHES_FILE).dropna(subset=['Team_A_ID', 'Team_B_ID'])
id_to_name = pd.concat([
    df_local[['Team_A_ID', 'Team_A_Name']].rename(columns={'Team_A_ID': 'ID', 'Team_A_Name': 'Name'}),
    df_local[['Team_B_ID', 'Team_B_Name']].rename(columns={'Team_B_ID': 'ID', 'Team_B_Name': 'Name'})
]).drop_duplicates(subset=['ID']).set_index('ID')['Name'].to_dict()

target_team_name = id_to_name.get(TARGET_TEAM_ID, f"Team {TARGET_TEAM_ID}")

# 2. Load AES Match Data
df_aes = pd.read_csv(AES_MATCHES_FILE)

# Standardize AES Results to boolean for easy math (Won = True, Lost = False)
df_aes['Is_Win'] = df_aes['Result'].astype(str).str.contains('Won', case=False, na=False)

# 3. Identify Target Team's Opponents
target_matches = df_aes[df_aes['TeamId'] == TARGET_TEAM_ID]
opponents_played = target_matches['Opponent'].dropna().unique()

target_wins = target_matches['Is_Win'].sum()
target_losses = len(target_matches) - target_wins
target_pct = (target_wins / len(target_matches) * 100).round(1) if len(target_matches) > 0 else 0

print(f"\nTarget Team: {target_team_name}")
print(f"Record: {target_wins}W - {target_losses}L ({target_pct}%)")
print(f"Unique Opponents Faced: {len(opponents_played)}")

# 4. Filter for OTHER Teams playing the SAME Opponents
# Get matches against the common opponents, but exclude matches played by our Target Team
common_matches = df_aes[
    (df_aes['Opponent'].isin(opponents_played)) & 
    (df_aes['TeamId'] != TARGET_TEAM_ID)
].copy()

# 5. Calculate Comparative Matrix
comparison = common_matches.groupby('TeamId').agg(
    Matches_Played=('Result', 'count'),
    Wins=('Is_Win', 'sum')
).reset_index()

# Math operations
comparison['Losses'] = comparison['Matches_Played'] - comparison['Wins']
comparison['Win_Pct'] = (comparison['Wins'] / comparison['Matches_Played'] * 100).round(1)

# 6. Formatting and Filtering
# Map the names
comparison['Team_Name'] = comparison['TeamId'].map(id_to_name).fillna("Unknown ID " + comparison['TeamId'].astype(str))

# Filter out noise (Teams that only played 1 or 2 common opponents don't provide a good baseline)
MIN_MATCHES = 3
comparison_filtered = comparison[comparison['Matches_Played'] >= MIN_MATCHES].copy()

# Sort by Win Percentage, then by Matches Played
comparison_filtered = comparison_filtered.sort_values(by=['Win_Pct', 'Matches_Played'], ascending=[False, False])

# Reorder columns for clean display
final_cols = ['Team_Name', 'Matches_Played', 'Wins', 'Losses', 'Win_Pct']
df_display = comparison_filtered[final_cols].reset_index(drop=True)

print(f"\n--- COMMON OPPONENT LEADERBOARD (Min {MIN_MATCHES} Matches) ---")
print("How other regional teams performed against the exact same opponents:")
display(df_display.head(20)) # Show top 20


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
FINISHES_FILE = Path('../Data/processed/aes_master_finishes.csv')
LOCAL_MATCHES_FILE = Path('../Data/processed/events/master_match_results.csv')

print("--- MISSION START: REGIONAL TEAM DOMINANCE REPORT ---")

# 1. Build an ID-to-Name Dictionary
df_local = pd.read_csv(LOCAL_MATCHES_FILE).dropna(subset=['Team_A_ID', 'Team_B_ID'])
id_to_name = pd.concat([
    df_local[['Team_A_ID', 'Team_A_Name']].rename(columns={'Team_A_ID': 'ID', 'Team_A_Name': 'Name'}),
    df_local[['Team_B_ID', 'Team_B_Name']].rename(columns={'Team_B_ID': 'ID', 'Team_B_Name': 'Name'})
]).drop_duplicates(subset=['ID']).set_index('ID')['Name'].to_dict()

# 2. Load the Finishes Database
if not FINISHES_FILE.exists():
    print(f"[!] Cannot find finishes database at {FINISHES_FILE}")
else:
    df_finishes = pd.read_csv(FINISHES_FILE)
    df_finishes['TeamId'] = df_finishes['TeamId'].astype(int)
    
    # 3. Aggregate Performance Metrics
    # We define a "Gold Bracket" finish as placing in the Top 25% of the field
    dominance = df_finishes.groupby('TeamId').agg(
        Tournaments_Played=('Tournament', 'count'),
        Avg_Top_Percent=('Top_Percent', 'mean'),
        First_Place_Finishes=('Rank', lambda x: (x == 1).sum()),
        Gold_Bracket_Finishes=('Top_Percent', lambda x: (x <= 25.0).sum())
    ).reset_index()
    
    # 4. Formatting and Calculations
    dominance['Team_Name'] = dominance['TeamId'].map(id_to_name).fillna("Unknown ID")
    
    # Round the average placement percentage
    dominance['Avg_Top_Percent'] = dominance['Avg_Top_Percent'].round(1)
    
    # Calculate how often they make the Gold Bracket
    dominance['Gold_Bracket_Rate'] = ((dominance['Gold_Bracket_Finishes'] / dominance['Tournaments_Played']) * 100).round(1).astype(str) + "%"
    
    # 5. Filtering and Sorting
    # Filter out teams with only 1 tournament to eliminate statistical noise
    MIN_TOURNAMENTS = 2
    dom_filtered = dominance[dominance['Tournaments_Played'] >= MIN_TOURNAMENTS].copy()
    
    # Sort by Average Top Percent ASCENDING (Lower % is better, e.g., Top 5% > Top 40%)
    # Tie-breaker goes to the team with more tournaments played
    dom_filtered = dom_filtered.sort_values(by=['Avg_Top_Percent', 'Tournaments_Played'], ascending=[True, False])
    
    # Organize columns for display
    final_cols = [
        'Team_Name', 
        'Tournaments_Played', 
        'Avg_Top_Percent', 
        'First_Place_Finishes', 
        'Gold_Bracket_Rate'
    ]
    df_display = dom_filtered[final_cols].reset_index(drop=True)
    
    print(f"\n--- REGIONAL TOURNAMENT LEADERBOARD (Min {MIN_TOURNAMENTS} Events) ---")
    print("* Note: A lower 'Avg_Top_Percent' indicates a higher tier of performance.")
    display(df_display.head(40)) # Display the Top 25 teams in the region


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# =============================================================================
# CONFIGURATION
# =============================================================================
TEAM_A_ID = 185651
TEAM_B_ID = 143614  # Your second target

AES_MATCHES_FILE = Path('../Data/processed/aes_master_matches.csv')
LOCAL_MATCHES_FILE = Path('../Data/processed/events/master_match_results.csv')

print(f"--- MISSION START: HEAD-TO-HEAD MUTUAL COMPARISON ---")

# 1. Load Data and Name Mapping
df_local = pd.read_csv(LOCAL_MATCHES_FILE).dropna(subset=['Team_A_ID', 'Team_B_ID'])
id_to_name = pd.concat([
    df_local[['Team_A_ID', 'Team_A_Name']].rename(columns={'Team_A_ID': 'ID', 'Team_A_Name': 'Name'}),
    df_local[['Team_B_ID', 'Team_B_Name']].rename(columns={'Team_B_ID': 'ID', 'Team_B_Name': 'Name'})
]).drop_duplicates(subset=['ID']).set_index('ID')['Name'].to_dict()

name_a = id_to_name.get(TEAM_A_ID, f"Team {TEAM_A_ID}")
name_b = id_to_name.get(TEAM_B_ID, f"Team {TEAM_B_ID}")

df_aes = pd.read_csv(AES_MATCHES_FILE)
df_aes['Is_Win'] = df_aes['Result'].astype(str).str.contains('Won', case=False, na=False)

# 2. Identify Mutual Opponents
opponents_a = set(df_aes[df_aes['TeamId'] == TEAM_A_ID]['Opponent'].dropna())
opponents_b = set(df_aes[df_aes['TeamId'] == TEAM_B_ID]['Opponent'].dropna())

mutual_opponents = list(opponents_a.intersection(opponents_b))

if not mutual_opponents:
    print(f"\n[!] No mutual opponents found between {name_a} and {name_b}.")
else:
    print(f"\nComparing {name_a} vs {name_b}")
    print(f"Shared Opponents Found: {len(mutual_opponents)}")

    # 3. Filter and Aggregate for the Summary Table
    def get_stats(team_id, opp_list):
        matches = df_aes[(df_aes['TeamId'] == team_id) & (df_aes['Opponent'].isin(opp_list))]
        return {
            'Played': len(matches),
            'Wins': matches['Is_Win'].sum(),
            'Losses': len(matches) - matches['Is_Win'].sum(),
            'Win_Pct': (matches['Is_Win'].sum() / len(matches) * 100).round(1) if len(matches) > 0 else 0
        }

    stats_a = get_stats(TEAM_A_ID, mutual_opponents)
    stats_b = get_stats(TEAM_B_ID, mutual_opponents)

    # 4. Construct Aggregate Comparison Table
    comparison_data = {
        'Metric': ['Matches vs Mutuals', 'Total Wins', 'Total Losses', 'Win Percentage'],
        name_a: [stats_a['Played'], stats_a['Wins'], stats_a['Losses'], f"{stats_a['Win_Pct']}%"],
        name_b: [stats_b['Played'], stats_b['Wins'], stats_b['Losses'], f"{stats_b['Win_Pct']}%"]
    }
    
    df_compare = pd.DataFrame(comparison_data)
    print("\n--- AGGREGATE PERFORMANCE AGAINST MUTUAL RIVALS ---")
    display(df_compare)

    # 5. Break it down by specific opponent (UPDATED LOGIC)
    detail_rows = []
    for opp in mutual_opponents:
        # Calculate Team A's exact record vs this opponent
        match_a = df_aes[(df_aes['TeamId'] == TEAM_A_ID) & (df_aes['Opponent'] == opp)]
        wins_a = match_a['Is_Win'].sum()
        losses_a = len(match_a) - wins_a
        record_a = f"{wins_a}W - {losses_a}L"
        
        # Calculate Team B's exact record vs this opponent
        match_b = df_aes[(df_aes['TeamId'] == TEAM_B_ID) & (df_aes['Opponent'] == opp)]
        wins_b = match_b['Is_Win'].sum()
        losses_b = len(match_b) - wins_b
        record_b = f"{wins_b}W - {losses_b}L"
        
        detail_rows.append({
            'Mutual Opponent': opp,
            f'{name_a} Record': record_a,
            f'{name_b} Record': record_b
        })
    
    df_detail = pd.DataFrame(detail_rows).sort_values(by='Mutual Opponent')
    print("\n--- DETAILED BREAKDOWN BY OPPONENT ---")
    display(df_detail.reset_index(drop=True))


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

# --- Configuration ---
LOCAL_MATCHES_FILE = Path('../Data/processed/events/master_match_results.csv')

print("--- BUILDING TEAM DIRECTORY ---")

if not LOCAL_MATCHES_FILE.exists():
    print(f"[!] Master match file not found at: {LOCAL_MATCHES_FILE}")
else:
    df_matches = pd.read_csv(LOCAL_MATCHES_FILE)
    df_matches = df_matches.dropna(subset=['Team_A_ID', 'Team_B_ID'])

    # Extract all unique ID/Name pairs from both 'A' and 'B' columns
    teams_a = df_matches[['Team_A_ID', 'Team_A_Name']].rename(columns={'Team_A_ID': 'TeamId', 'Team_A_Name': 'TeamName'})
    teams_b = df_matches[['Team_B_ID', 'Team_B_Name']].rename(columns={'Team_B_ID': 'TeamId', 'Team_B_Name': 'TeamName'})

    # Combine, drop duplicates, and sort
    df_directory = pd.concat([teams_a, teams_b]).drop_duplicates().sort_values(by='TeamName').reset_index(drop=True)
    df_directory['TeamId'] = df_directory['TeamId'].astype(int)

    print(f"[SUCCESS] Directory built with {len(df_directory)} unique teams.")

    # --- Interactive Search Loop ---
    while True:
        print("\n" + "-"*50)
        query = input("Enter a team name to search (or type 'EXIT' to quit): ").strip()

        if query.upper() == 'EXIT':
            break

        if not query:
            # If the user enters nothing, show the full directory
            print("\n--- FULL TEAM DIRECTORY ---")
            with pd.option_context('display.max_rows', None): # Ensure all rows are shown
                 display(df_directory)
        else:
            # Perform a case-insensitive search
            results = df_directory[df_directory['TeamName'].str.contains(query, case=False, na=False)]

            if results.empty:
                print(f"\n[!] No teams found matching '{query}'.")
            else:
                print(f"\n--- SEARCH RESULTS FOR '{query}' ---")
                display(results)

print("\n--- Exited Team Directory ---")

